# pandas와 연구 데이터셋

## Bike Sharing 자료의 구조와 품질 점검

시간별 자전거 대여자료를 읽고 관측 단위, 식별자, 자료형, 결측, 관계식과 실제 파일 범위를 확인합니다.


## 이번 실습에서 완성할 것

- 첫 행·열·자료형을 확인합니다.
- 관측 단위와 식별자, 결측·중복·관계식을 점검합니다.
- 두 시간대의 요약통계를 같은 단위로 비교합니다.

**완료 결과:** 자료 구조표 1개, 품질점검 결과와 두 시간대 비교문장


## 이번 교시에서 사용할 데이터

### 강의 시연: OWID CO₂ 수업용 자료

| 항목 | 내용 |
|---|---|
| 출처 | Our World in Data CO₂ 공개자료에서 수업 범위만 추출한 snapshot |
| 규모 | 175행, 5개 지역, 1990~2024년 |
| 한 행의 의미 | 한 지역의 한 연도 CO₂ 배출 관측 |
| 주요 열 | `entity`, `code`, `year`, `emissions_total`, `emissions_total_per_capita` |
| 단위 | 총배출량은 tonnes, 1인당 배출량은 tonnes/person |

`World`는 국가가 아니라 집계지역입니다. 국가 간 비교를 할 때 포함 여부를 먼저 결정해야 하며, 총배출량과 1인당 배출량은 서로 다른 연구질문에 답합니다. 이 파일은 전체 OWID 자료가 아니라 다섯 지역만 포함한 교육용 부분집합입니다.

### 적용 실습: UCI Bike Sharing

| 항목 | 내용 |
|---|---|
| 출처 | UCI Machine Learning Repository의 Bike Sharing Dataset |
| 규모 | 17,379행, 2011~2012년 시간별 기록 |
| 한 행의 의미 | 특정 날짜의 특정 시간대에 기록된 대여수요와 날씨·달력 조건 |
| 주요 열 | `dteday`, `hr`, `season`, `workingday`, `weathersit`, `temp`, `hum`, `casual`, `registered`, `cnt` |

`season`, `weekday`, `weathersit`는 숫자로 저장된 범주입니다. 또한 `cnt = casual + registered`이므로 `cnt`를 예측할 때 두 구성 열을 feature로 사용하면 결과 누수가 발생합니다.


## 이번 실습의 분석 순서

Bike Sharing 자료를 읽고, 한 행의 의미와 key를 정한 뒤 자료형·결측·중복·관계식을 점검합니다. 기본 실행에서는 전체 자료를 감사하고, 이후 `SELECTED_HOUR` 하나만 바꾸어 선택조건에 따른 요약통계의 차이를 확인합니다.

**남길 결과:** 데이터 구조 감사표, 선택시간 요약표, 분석 명세, 해석 한계 한 문장


## 실습 구간 안내

| 구분 | 수행 범위 |
|---|---|
| **수업 중 필수** | 데이터 읽기부터 분석 명세·PASS까지 |
| **시간이 남으면** | 8시와 17시 요약통계 비교 |
| **수업 후 확장(선택)** | 분석용 데이터사전과 실행 가능한 계약 |

수업 중에는 필수 구간의 결과와 마지막 `PASS`를 먼저 완성합니다. 확장 구간을 실행하지 않아도 필수 셀이 독립적으로 작동하도록 구성되어 있습니다.


## 1. 데이터셋을 실행 전에 읽기

UCI Bike Sharing 자료는 Capital Bikeshare의 2011~2012년 시간별 대여 건수와 날씨·달력 정보를 담습니다. 공식 페이지는 17,389 instances와 결측 없음으로 표시합니다. 실제 파일을 열기 전에 다음을 예상합니다.

- 한 행은 무엇인가?
- 한 행을 고유하게 찾을 key는 무엇인가?
- 숫자로 저장됐지만 범주인 열은 무엇인가?
- `cnt`, `casual`, `registered`는 어떤 관계인가?

이 질문의 첫 답은 가설입니다. 실행 결과와 `0교시에서 내려받은 자료와 공식 설명`로 확인합니다.


## 2. 실행 환경과 데이터 경로 확인


In [ ]:
from pathlib import Path
import platform

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 120)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행해 주세요."
    )

DATA_PATH = find_course_data("bike/hour.csv")
print("Python:", platform.python_version())
print("data:", DATA_PATH.relative_to(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())))


## 3. DataFrame을 읽고 첫 관측 확인

`pd.read_csv`는 CSV 파일을 DataFrame으로 읽습니다. `parse_dates`는 날짜 열을 문자 대신 날짜 자료형으로 해석합니다.

`shape`의 첫 숫자는 행 수, 두 번째 숫자는 열 수입니다. `head(3)`은 첫 세 행을 보여줄 뿐 전체 자료의 대표 표본은 아닙니다.


In [ ]:
bike = pd.read_csv(DATA_PATH, parse_dates=["dteday"])

print("object type:", type(bike).__name__)
print("shape (rows, columns):", bike.shape)
display(bike.head(3))
print("\n[첫 관측을 세로로 읽기]")
print("이 한 행은 특정 날짜·시간대의 날씨, 달력 조건과 대여건수를 나타냅니다.")
display(bike.iloc[[0]][["dteday", "hr", "workingday", "weathersit", "temp", "hum", "casual", "registered", "cnt"]].T.rename(columns={bike.index[0]: "첫 행의 값"}))


**기록:** 공식 metadata의 예상 행 수와 실제 `shape`가 같은가? 다르면 어느 값을 기준으로 분석할지 적어 봅니다.


## 4. Series와 DataFrame 선택


In [ ]:
rental_count = bike["cnt"]
hourly_table = bike[["dteday", "hr", "cnt"]]

print('bike["cnt"]:', type(rental_count).__name__, rental_count.shape)
print('bike[["dteday", "hr", "cnt"]]:', type(hourly_table).__name__, hourly_table.shape)
display(hourly_table.head(3))


열 이름 하나를 쓰면 Series, 열 이름 목록을 쓰면 DataFrame이 됩니다. 분석 함수가 1차원과 2차원 중 어느 입력을 요구하는지에 따라 선택 방법이 달라집니다.


## 5. 열 구조 감사

숫자로 저장됐다고 모두 연속형 measure는 아닙니다. `season`, `weekday`, `weathersit`은 정수 code로 표현된 category입니다. 반대로 `temp`와 `hum`은 실수지만 원래 단위를 정규화한 값입니다.


In [ ]:
role_map = {
    "instant": "identifier",
    "dteday": "dimension: date",
    "season": "dimension: category code",
    "yr": "dimension: category code",
    "mnth": "dimension: category code",
    "hr": "dimension: hour",
    "holiday": "dimension: binary",
    "weekday": "dimension: category code",
    "workingday": "dimension: binary",
    "weathersit": "dimension: category code",
    "temp": "measure: normalized",
    "atemp": "measure: normalized",
    "hum": "measure: normalized",
    "windspeed": "measure: normalized",
    "casual": "measure: count component",
    "registered": "measure: count component",
    "cnt": "measure: total count",
}

structure_audit = pd.DataFrame({
    "role": pd.Series(role_map),
    "dtype": bike.dtypes.astype(str),
    "missing": bike.isna().sum(),
    "unique": bike.nunique(),
})
display(structure_audit)

print("date range:", bike["dteday"].min().date(), "to", bike["dteday"].max().date())
print("hour range:", bike["hr"].min(), "to", bike["hr"].max())


**기록:** `dtype`만 보고 범주와 측정값을 완전히 구분할 수 없는 이유를 한 문장으로 적어 봅니다.


## 6. key, 결측, 중복, 관계식 확인

key 후보는 설명서에 적혀 있다는 이유만으로 채택하지 않습니다. 실제 파일에서 중복을 검사합니다. `cnt = casual + registered`도 모든 행에서 성립하는지 확인합니다.


In [ ]:
key_checks = pd.Series({
    "all missing cells": int(bike.isna().sum().sum()),
    "duplicated full rows": int(bike.duplicated().sum()),
    "duplicated instant": int(bike.duplicated(["instant"]).sum()),
    "duplicated date + hour": int(bike.duplicated(["dteday", "hr"]).sum()),
    "rows violating cnt = casual + registered": int(
        bike["cnt"].ne(bike["casual"] + bike["registered"]).sum()
    ),
}, name="count")
display(key_checks.to_frame())
print("\n[검사 결과를 문장으로 읽기]")
for check_name, value in key_checks.items():
    print(f"- {check_name}: {value}")
print("date + hour 중복이 0이면 각 시간 관측을 그 조합으로 구분할 수 있습니다.")
print("관계식 위반이 0이면 이 파일에서 cnt = casual + registered가 성립합니다.")


`cnt`를 나중에 예측한다면 `casual`과 `registered`는 정답을 구성하는 정보입니다. 예측 시점에 이 값을 입력으로 쓰면 target leakage가 됩니다.


## 7. metadata와 실제 파일 비교


In [ ]:
metadata_comparison = pd.DataFrame({
    "source": ["UCI web metadata", "distributed hour.csv"],
    "reported_or_observed_rows": [17_389, len(bike)],
})
metadata_comparison["difference_from_file"] = (
    metadata_comparison["reported_or_observed_rows"] - len(bike)
)
display(metadata_comparison)


metadata와 파일이 다를 때 한쪽을 조용히 수정하지 않습니다. 원출처, 접근일, 실제 사용한 파일, 차이, 이후 분석 기준을 함께 보고합니다.


## 8. 조건 선택과 작은 변형

`bike["hr"].eq(8)`은 각 행이 8시 관측인지 True/False로 판단합니다. `loc[조건, 열 목록]`은 True인 행과 필요한 열만 선택합니다.

첫 실행 후 `FOCUS_HOUR = 8`을 `17`로 바꾸어 다시 실행합니다. 코드 전체를 다시 쓰지 않습니다.


In [ ]:
# 이 값만 8에서 17로 바꾸어 같은 검사를 다시 실행해 보십시오.
FOCUS_HOUR = 8

focus = bike.loc[
    bike["hr"].eq(FOCUS_HOUR),
    ["dteday", "hr", "workingday", "weathersit", "cnt"],
].copy()

print("selected hour:", FOCUS_HOUR)
print("selected rows:", len(focus))
display(focus.head(5))
display(focus["cnt"].describe().to_frame(name="rental count"))


## 시간이 남으면 — 8시와 17시 수요의 요약통계 비교

평균만 비교하지 않고 중앙값·표준편차·사분위수와 관측 수를 함께 봅니다. 두 시간대 중 평균과 중앙값의 차이가 더 큰 곳을 찾아 그 이유에 대한 가설을 한 문장으로 기록하세요.


In [ ]:
hour_comparison = (
    bike.loc[bike["hr"].isin([8, 17])]
    .groupby("hr")["cnt"]
    .agg(
        observations="size",
        mean="mean",
        median="median",
        std="std",
        minimum="min",
        q25=lambda s: s.quantile(0.25),
        q75=lambda s: s.quantile(0.75),
        maximum="max",
    )
)
hour_comparison["mean_minus_median"] = hour_comparison["mean"] - hour_comparison["median"]
display(hour_comparison.round(2))
print("평균과 중앙값의 차이가 큰 시간대:", hour_comparison["mean_minus_median"].abs().idxmax())


## 수업 후 확장(선택) — 분석용 데이터사전과 실행 가능한 계약

필수 실습을 마친 뒤 수행합니다. 분석에 사용할 열의 의미·단위·역할을 표로 남기고, 이 문서가 실제 데이터와 일치하는지 코드로 확인합니다. `casual`과 `registered`는 `cnt`를 구성하므로 예측 feature에서 제외합니다.


In [ ]:
dictionary_rows = [
    {"column": "dteday", "meaning": "관측 날짜", "unit_or_level": "date", "analysis_role": "date-hour key"},
    {"column": "hr", "meaning": "관측 시간", "unit_or_level": "0–23 hour", "analysis_role": "key and candidate feature"},
    {"column": "workingday", "meaning": "근무일 여부", "unit_or_level": "0/1", "analysis_role": "candidate feature"},
    {"column": "weathersit", "meaning": "날씨 등급", "unit_or_level": "1–4 category", "analysis_role": "candidate feature"},
    {"column": "temp", "meaning": "정규화 기온", "unit_or_level": "normalized", "analysis_role": "candidate feature"},
    {"column": "hum", "meaning": "정규화 습도", "unit_or_level": "normalized", "analysis_role": "candidate feature"},
    {"column": "cnt", "meaning": "시간당 총 대여건수", "unit_or_level": "rentals/hour", "analysis_role": "target"},
    {"column": "casual", "meaning": "비등록 이용자 대여건수", "unit_or_level": "rentals/hour", "analysis_role": "exclude: target component"},
    {"column": "registered", "meaning": "등록 이용자 대여건수", "unit_or_level": "rentals/hour", "analysis_role": "exclude: target component"},
]
data_dictionary = pd.DataFrame(dictionary_rows)
data_dictionary["dtype_in_file"] = data_dictionary["column"].map(
    lambda column: str(bike[column].dtype) if column in bike.columns else "MISSING"
)
print("[분석용 데이터사전]")
display(data_dictionary)

candidate_features = ["hr", "workingday", "weathersit", "temp", "hum"]
contract_checks = pd.Series({
    "dictionary columns exist in file": set(data_dictionary["column"]).issubset(bike.columns),
    "date-hour key is unique": not bike.duplicated(["dteday", "hr"]).any(),
    "target and its components are excluded from features": set(candidate_features).isdisjoint(
        {"cnt", "casual", "registered"}
    ),
}, name="PASS")
print("[데이터사전과 분석설정의 계약 검사]")
display(contract_checks.to_frame())
assert bool(contract_checks.all()), "데이터사전 또는 분석설정이 실제 파일과 일치하지 않습니다."
print("데이터사전의 열과 feature 사용 규칙: PASS")


**비교 기록:** 8시와 17시의 행 수, 평균, 중앙값이 어떻게 달랐는가? 이 차이만으로 시간대의 인과효과를 말할 수 있는가?


## 9. 데이터 분석 명세 생성


In [ ]:
data_contract = pd.DataFrame({
    "item": [
        "research object", "grain", "key", "dimensions",
        "measures", "time coverage", "known warning",
    ],
    "record": [
        "Capital Bikeshare hourly operation",
        "date × hour",
        "instant; verified again with dteday + hr",
        "date, hour, weekday, working day, season, weather category",
        "normalized weather variables and hourly rental counts",
        f"{bike['dteday'].min().date()} to {bike['dteday'].max().date()}",
        "metadata row count differs from file; casual + registered = cnt",
    ],
})
display(data_contract)


| 항목 | OWID CO₂ | UCI Bike Sharing |
|---|---|---|
| 연구 대상 |  |  |
| grain |  |  |
| key |  |  |
| 시간 해상도 |  |  |
| 핵심 measure와 단위 |  |  |
| 분석 전 주의점 |  |  |


## 10. 결과 기록과 ChatGPT 검토

다음 셀은 `key_checks`, `metadata_comparison`, `data_contract`와 선택 시간의 요약통계를 직접 넣어 검토 프롬프트를 만듭니다.

- **관찰:** 표에서 직접 읽은 행 수·결측·중복·요약통계
- **조건:** 자료 버전, 한 행의 의미, key, `FOCUS_HOUR`
- **한계:** metadata 차이, 관계식 누수와 현재 감사로 확인하지 못한 내용


In [ ]:
focus_summary_for_prompt = focus["cnt"].describe().to_frame("rental_count")
review_prompt = f"""
Bike Sharing 연구자료 감사를 다음 실제 출력만 사용해 검토해 주세요.

- 파일 shape: {bike.shape}
- 선택 시간: {FOCUS_HOUR}, 선택 행 수: {len(focus)}
- key·결측·관계식 검사:
{key_checks.to_string()}
- metadata와 실제 파일 비교:
{metadata_comparison.to_string(index=False)}
- 선택 시간 요약통계:
{focus_summary_for_prompt.to_string()}
- 데이터 분석 명세:
{data_contract.to_string(index=False)}

관찰 | 분석조건 | 해석한계의 세 항목으로 답해 주세요. 각 관찰에는 위 객체의
열 이름과 실제 값을 쓰고, metadata 차이와 casual + registered = cnt 관계가
향후 예측에서 만드는 누수 위험을 구분해 주세요. 표에 없는 값을 만들지 마세요.
""".strip()
print("[ChatGPT에 복사할 교시별 검토 프롬프트]\n")
print(review_prompt)


## 11. 실행 완료 확인

다음 셀은 연구 결론의 타당성을 판정하지 않습니다. 필요한 객체와 기본 감사가 실행됐는지만 확인합니다.


In [ ]:
completion_checks = pd.Series({
    "data loaded": isinstance(bike, pd.DataFrame) and len(bike) == 17_379,
    "date parsed": str(bike["dteday"].dtype).startswith("datetime64"),
    "structure audited": structure_audit.shape[0] == bike.shape[1],
    "key checked": key_checks["duplicated date + hour"] == 0,
    "identity checked": key_checks["rows violating cnt = casual + registered"] == 0,
    "focus hour valid": FOCUS_HOUR in range(24),
}, name="passed")
display(completion_checks.to_frame())
assert completion_checks.all(), "통과하지 못한 항목의 이전 셀을 다시 확인해 주세요."
print("자료 구조 검증: PASS")


## 다음 교시 연결

이번 시간에는 원자료의 grain을 확인했습니다. 다음 시간에는 `groupby`, `pivot`, `melt`, `merge`를 사용하면서 표를 바꿀 때 grain이 어디에서 달라지는지 추적합니다.
